# Text Analytics Lab 2: Word Embeddings and Topic Models 

This lab explores unsupervised methods for learning from text data. Firstly, we use the text representations from the previous lab to identify topics with Latent Dirichlet Allocation (LDA), then show how to adjust term count vectors with TF-IDF. Next, we introduce embedding techniques for learning dense vector representations of words and documents. Building on this idea, we show how to use BERTopic, which uses the idea of learned embeddings to cluster documents.

### Learning Outcomes

After completing this lab, you should be able to...
* Apply Latent Dirichlet Allocation (LDA) and BERTopic to identify topics
* Interpret the outputs of LDA and BERTopic for a specific document or topic 
* Use TF-IDF to improve the vector representations of the documents and recognise its limitations.
* Train simple word embedding models and retrieve embeddings for specific words. 

### Outline

* Implementing LDA Topic Modelling
* TF-IDF vectors
* Visualizing Topic Modelling Results
* Word embeddings
* BERTopic
* Optional: HDP model as an alternative to LDA

## 1: Load the Dataset

Faced with large collection of documents, it is often useful to understand the different themes or topics that are being discussed. For example, what topics are covered in a set of news articles? What issues to customers often complain about in their reviews? 

Topic modelling uses unsupervised learning to extract topics from a collection of documents. Each topic is represented as a probability distribution over words: for example, for a topic related to "space travel" you would expect words like "shuttle" or "moon" to have a high probability, while unrelated words such as "reggae" or "hippopotamus" will have low probability. 

Let's load some data to apply topic modelling to from the 20 newsgroups dataset:

In [ ]:
from sklearn.datasets import fetch_20newsgroups

# We will use the 'train' split for learning the topics in an unsupervised manner:
newsgroups_train = fetch_20newsgroups(subset='train', shuffle=True, remove=('headers', 'footers', 'quotes'))

# We will apply our learned topic model to the 'test' split later on:
newsgroups_test = fetch_20newsgroups(subset='test', shuffle=True, remove=('headers', 'footers', 'quotes'))

Newsgroups are internet discussion groups, where users discuss a range of topics, which were popular in the early days of the Internet. Despite the name, the posts do not usually contain news. This dataset contains posts from 20 different newsgroups. Each newsgroup has a particular theme. We can view the list of newsgroups in the training split as follows:

In [ ]:
print(newsgroups_train.target_names)

When working with a new dataset, it is always good to play around with the data and see what we have, and how to find everything :) 
```newsgroups_train``` is a Python dictionary. Let's look at the keys and the types of objects stored in the dictionary:

In [ ]:
for i in newsgroups_train:
    print(i, type(newsgroups_train[i]))

The ```data``` item is a list of raw text documents. This is all we need for topic modelling, so we can ignore the ```target``` and ```DESCR``` keys for now.

The code below prints out the first post in ```data```:

In [ ]:
print('DOCUMENT 0:')
print(newsgroups_train.data[0])

## 2: Data Preprocessing

To apply topic modelling, we need to first preprocess the data. We will carry out the following steps:
* Tokenise the posts using NLTK's word_tokenize() function. Topic models are a way of clustering words, so we have to split our text into sequences of words. 
* Remove non-word tokens and tokens with length less than 3 (likely to be numbers and punctuation that are not related to specific topics) and longer than 15 (probably URLs, codes, andbadly formatted tokens rather than proper words). This is a heuristic for cleaning the data.
* Convert the tokens to lower case. This simplifies the vocabulary.
* Remove stopwords: we have not used this step before; it removes tokens such as 'the' and 'a' that appear in a list of very common words, because these words do not tell us much about topics. Stopword removal is not always a good idea, as it can sometimes remove important information, so be careful about when you apply it! For topic modelling, removing stopwords can be  useful to remove noise and reduce the model complexity. 
* Lemmatize the tokens using WordNetLemmatizer to convert verbs to their root forms. This also minimises the vocabulary size. We assume that if a verb is associated with a topic, then all forms of the verb will share this association.  

We're going to use the library [Gensim](https://radimrehurek.com/gensim/), because it contains a lot of useful tools for topic modelling. The preprocessing steps will therefore look a bit different to our first lab, where we used Scikit-learn, as Gensim also comes with useful functions for text normalisation, listing stopwords, and representing words or documents as vectors. Run the code below to preprocess the text:

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from nltk.stem import WordNetLemmatizer 
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS # find stopwords
import numpy as np

np.random.seed(400)  # We fix the random seed to ensure we get consistent results when we repeat the lab.

# Tokenize and lemmatize
def preprocess(text):
    result=[]
    for token in simple_preprocess(text) :  # Tokenize, remove very short and very long words, convert to lower case, remove words containing non-letter characters
        if token not in STOPWORDS:
            result.append(WordNetLemmatizer().lemmatize(token, 'v'))
            
    return result

# Create a list of preprocessed documents
processed = []
for doc in newsgroups_train.data:
    processed.append(preprocess(doc))

Now that we have finished the preprocessing, we need to construct the input for Gensim's topic modelling method. We do so by constructing a dictionary with word<->id mappings, then converting that into a bag of words, which will be the input to our model.

In [ ]:
from gensim.corpora import Dictionary

dictionary = Dictionary(processed) # construct word<->id mappings - it does it in alphabetical order
print(dictionary)

bow_corpus = [dictionary.doc2bow(doc) for doc in processed]

# 3. Latent Dirichlet Allocation (LDA)

Now we are ready to perform topic modelling using LDA. 

We are going to try 20 topics in the document corpus: the number of newsgroups is also 20, so perhaps we will find topics that correspond with the newsgroups. We will be running LDA using all CPU cores to parallelize and speed up model training.

Gensim provides the ```LdaModel``` class. When we construct an ```LdaModel``` object, some of the parameters we will be tweaking are:
   * *num_topics*, the number of requested latent topics to be extracted from the training corpus. <br>
   * *id2word*, a mapping from word ids (integers) to words (strings). It is used to determine the vocabulary size, as well as for debugging and topic printing. <br>
   * *workers*, the number of extra processes to use for parallelization. Uses all available cores by default. <br>


In [ ]:
from gensim.models import LdaModel

# This call will construct and fit (train) the LDA model:
lda_model =  LdaModel(bow_corpus, 
                      num_topics=20, 
                      id2word=dictionary,                                    
                      passes=10,
                    ) 

**TO-DO 1:** Run the code below to print out the topic distributions found by LDA. Can you find any topics that relate to specific newsgroups (newsgroups_train.target_names)? Can you find any other meaningful topics?

In [ ]:
'''
For each topic, we will explore the words occuring in that topic and its relative weight
'''
for idx, topic in lda_model.print_topics(-1):
    print("Topic: {} \nWords: {}".format(idx, topic ))
    print("\n")

**TO-DO 2:** What do the values beside each word mean? 



## Testing on unseen data

Now that we have a trained LDA model, we can run it on a new, unseen document to get the breakdown of topics. Let's test the model on a document from the test set. First, get the raw document, then apply preprocessing:

In [ ]:
# Set the index below to choose a document:
test_document_idx = 0

# Retrieve the document and print its newsgroup:
unseen_document = newsgroups_test.data[test_document_idx]
print(unseen_document)

print(f' This document is from newsgroup {newsgroups_test.target_names[newsgroups_test.target[test_document_idx]]}')

# Data preprocessing step for the unseen document - It is the same preprocessing we have performed for the training data
bow_vector = dictionary.doc2bow(preprocess(unseen_document))

# Show a bag of words representation of the object:
for idx, count in bow_vector:
    print(f'{dictionary[idx]}: {count}')

Now, run our preprocessed document through the LDA model as follows to obtain $\boldsymbol{\theta}^d$, the topic distribution (topics with zero probability are not shown):

In [ ]:
topic_distribution = lda_model[bow_vector]

We can examine the word-topic distributions for the topics associated with this document:

In [ ]:
for index, probability in sorted(topic_distribution, key=lambda tup: -1*tup[1]):
    print("Index: {}\nProbability: {}\t Topic: {}".format(index, probability, lda_model.print_topic(index, 5)))

**TO-DO 3:** Do these topics seem like a good fit for your selected document?

## Visualising Topics 

Let's compare the topics for some training set documents in the same newsgroup.

**TO-DO 4:** Complete the code below to define a function to retrieve the topic distributions for 10 documents in the ```talk.politics.mideast``` newsgroup.

In [ ]:
from gensim.matutils import any2sparse

def get_document_ids_in_newsgroup(newsgroup_name, newsgroups_data):
    # retrieve a list of document indexes for documents with this target_name
    doc_idxs = []
    for i, target_i in enumerate(newsgroups_data.target):
        if newsgroups_data.target_names[target_i] == newsgroup_name:
            doc_idxs.append(i)
            
    #print("There are {} documents in the newsgroup {}".format(len(doc_idxs), newsgroup_name))
            
    return doc_idxs


def get_topic_dists_in_newsgroup(newsgroup_name, lda_model, max_num_docs=10):
    doc_idxs = get_document_ids_in_newsgroup(newsgroup_name, newsgroups_train)
    
    # only use the first ten documents.
    if len(doc_idxs) > max_num_docs:
        doc_idxs = doc_idxs[:max_num_docs]
    print(doc_idxs)
    
    # Save each theta_d distribution to the list 'thetas':
    thetas = []
    
    for doc_idx in doc_idxs:
        ### COMPLETE THE CODE HERE
        

        
        #######################
        
        thetas.append(theta_d)
    
    return thetas

Run the function above and print out the topic distributions:

In [ ]:
thetas = get_topic_dists_in_newsgroup('talk.politics.mideast', lda_model, max_num_docs=10)
print(thetas)

The text output is quite hard to read. We can improve on it by making a bar chart with a distinct colour for each topic. The function below can be used to plot a simple bar chart using the ```matplotlib``` library. This will help us see which documents discuss the same topics. The height of each bar is the probability for that topic within the document, according to our model.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# choose some colours for the topics
colours = ['blue', 'green', 'red', 'cyan', 'magenta', 'yellow', 'black', 'teal', 'pink', 'purple',
           'orange', 'gray', 'lime', 'darkgreen', 'lightgray', 'navy', 'gold', 'crimson', 'darkgray', 'fuchsia']

def convert_theta_sparse_to_dense(theta_d_sparse, num_topics):
    theta_d = np.zeros(num_topics)  # an empty array
    
    # split the output from lda_model into two lists
    active_topics_in_d, probs = map(list, zip(*theta_d_sparse))
    
    # record the values in theta_d
    for i, topic in enumerate(active_topics_in_d):
        if topic >= num_topics:
            break
            
        theta_d[topic] = probs[i]
    
    return theta_d

# a function for producing a bar chart for a document
def plot_theta(thetas, d, num_docs, num_topics):
    plt.subplot(int(num_docs/3) + 1, 3, d+1)   # make a set of subplots inside a figure, with four subplots per row
    
    theta_d = convert_theta_sparse_to_dense(thetas[d], num_topics)
    
    # plot the results so that the same topics always occur at the same place along the x axis.
    plt.bar(x=np.arange(len(theta_d)), height=theta_d, color=colours, tick_label=np.arange(num_topics))

**TO-DO 5:** Use the function to plot the topic distributions of the 10 documents we selected from the ```talk.politics.mideast``` newsgroup. What do you notice? Are there any topics the documents have in common? Any they do not? Refer back to the printed list of topics above to find the most common topic in this newsgroup.

ANSWER: write your answer here.

In [ ]:
plt.figure(figsize=(20,8))

### WRITE YOUR OWN CODE HERE


############################

plt.show()

Now, let's look at how the topics we've found with LDA relate to the newsgroups (the targets in the training set).

The code below iterates over the newsgroups, computing the topic distribution for each document in that particular newsgroup. 

**TO-DO 6:** Complete the function below to compute the mean topic distribution, $\boldsymbol\theta$, for each newsgroup. Hint: use the ```mean``` function from Numpy. 

In [ ]:
# Create a numpy array which will store a matrix of values. Rows correspond to newsgroups and columns to LDA topics.
def get_newsgroups_mean_topics(lda_model, num_topics):
    
    # Create a matrix where each row corresponds to a newsgroup, and each column to a topic.
    # In each entry, we will save the mean probability of the topic for the documents in that newsgroup.
    mean_thetas = np.zeros((len(newsgroups_train.target_names), num_topics))

    print(mean_thetas.shape)
    for t, target_name in enumerate(newsgroups_train.target_names):
        # Obtain the thetas for the documents with this target name
        thetas_t_sparse = get_topic_dists_in_newsgroup(target_name, lda_model, max_num_docs=10)
        
        print(thetas_t_sparse)
        
        # convert the thetas to a dense vector format
        thetas_t = []
        for theta_d_t_sparse in thetas_t_sparse:
            if not theta_d_t_sparse:
                continue  # if it's empty
            thetas_d = convert_theta_sparse_to_dense(theta_d_t_sparse, num_topics)
            thetas_t.append(thetas_d)
            
        # compute the mean theta for this newsgroup and store it in mean_thetas 
        
        ### WRITE YOUR OWN CODE HERE
        
        ###########################

        #print(mean_thetas[t])
    return mean_thetas

We can now plot a matrix ```mean_thetas``` using ```matplotlib``` using the code below. This shows how much of the text within each newsgroup relates to each LDA topic. So, a more yellow square means that the newsgroup is more strongly associated with that topic. 

**TO-DO 7:** which LDA topics are common across many newsgroups? Are there any topics that are specific to particular newsgroups? 

ANSWER: write your answer here.

In [ ]:
from IPython.display import clear_output

def plot_newsgroup_topic_matrix(model, num_topics):
    # Run the function we defined above to get the mean topic distributions:
    mean_thetas = get_newsgroups_mean_topics(model, num_topics)
    
    print(f'mean_thetas is a matrix of shape {mean_thetas.shape}')

    # Create a new figure
    plt.figure(figsize=(20,10))
    clear_output()

    # Plot the matrix as a 2-D grid, where colours represent the values.
    plt.imshow(mean_thetas)

    # Change the labels on the axes
    plt.yticks(range(len(newsgroups_train.target_names)), newsgroups_train.target_names )
    plt.xticks(range(num_topics))

    plt.show()
    
plot_newsgroup_topic_matrix(lda_model, num_topics)

# 4. TF-IDF

So far for LDA we have used a bag of words to represent each document. Despite removing stopwords, the unigram feature counts can still emphasise common words that are not very helpful for distinguishing between topics. To see this, let's take a document from the 'rec.autos' newsgroup as a 'query' document and print out the document and its bag of words vector:

In [ ]:
doc_idxs = get_document_ids_in_newsgroup('rec.autos', newsgroups_train)
doc_idx_0 = doc_idxs[19]
print(newsgroups_train.data[doc_idx_0].strip())
query_doc = bow_corpus[doc_idx_0]
# show the bag of words vector in sparse format. In each pair of numbers, the first is the word ID and the second is the word count.

query_doc.sort(key=lambda tup: -1*tup[1])  # sort the bag of words vector for the query document in descending order of word count
for idx, count in query_doc:  
    print(f'{dictionary[idx]}: {count}')

In some cases, we can improve the vector representations from a pure bag of words using TF-IDF to focus more on keywords that distinguish one document from the others. Our usual bag of words vector contains word counts -- these values are otherwise known as 'term frequencies' or TF.

TF-IDF is computed from two terms. First, the log of the term frequency:
$$ tf(t,d) = count(t,d)$$

Term frequency captures how much that term occurs in a document, as a way of capturing its importantance to that document. However, we now want to down-weight terms that are common in every document so that the distinctive keywords have the highest values. To do this, we multiple the log term frequency by the inverse document frequency, which is computed like this:

$$ idf(t) = \log_{2}\frac{N}{df(t)}$$

Document frequency captures how many documents the term occurs in, thus measuring how common the term is across the whole dataset. By taking an inverse, terms with high document frequency will be down-weighted. 

Gensim provides the TfidfModel class to compute TF-IDF vectors from our existing corpus object:

In [ ]:
from gensim.models import TfidfModel

tfidf_model = TfidfModel(bow_corpus)

**TO-DO 8:** Apply the tfidf_model to `query_doc`, our randomly selected document from the newsgroups dataset. Print the results to see how the values differ from the previous bag of words. Hint: you can apply the model in the same was a you applied lda_model to a bow_vector for an unseen test document.

In [ ]:
#### WRITE YOUR OWN CODE HERE


#####

**TO-DO 9:** Rerun LDA using TF-IDF vectors and print the topic-word distributions. Do you notice any differences in the topics?

In [ ]:
tfidf_corpus = [tfidf_model[doc] for doc in bow_corpus]

### WRITE YOUR ANSWER HERE


    
######

# 5. Word Embeddings

In this section, you will explore word embeddings: vector representations that have themselves been learned from unlabelled text data. These embeddings encode information about a word's meaning, making them useful for representing text as input to a topic model or classifier. 

First, consider the bag-of-words and TF-IDF vectors above. THey are sparse, with many zeros and a dimension for each term in the vocabulary. They do not capture complex relations between words very well, e.g., synonyms are treated as unrelated. _Word embeddings_ address this problem by mapping each word to a vector representation. Documents can then be represented by aggregating word vectors, or we can pass sequences of word embeddings into neural networks (coming up in future weeks!). 

This idea became successful afeter Mikolov et al. published a paper on their ``word2vec`` package in 2013. Importantly, the word embeddings can be learned from lots of text without any labelling, and learn to capture general aspects of semantics (meaning) and syntax that apply to many different problems. The word2vec approach is to train classifiers to predict how frequently pairs of words occur in the same context, i.e., within a few words of each other in a document in the training set. For each word in the vocabulary, we learn a logistic regression classifier, and its weights can be used as a vector that represents the word. This weight vector is a by-product of the training task that happens to encode a lot of useful information about word semantics: words that occur in similar contexts typically have some kind of relationship, and will end up with similar weight vectors. 

Let's first use Gensim to train a word2vec model. The code below tokenizes the training texts, then runs word2vec (the _skipgram_ variant) to learn a set of embeddings. 

In [ ]:
from gensim.models import word2vec
from gensim.utils import tokenize

raw_text = newsgroups_train.data  # get the training documents as a text corpus for learning embeddings

tokenized_texts = [list(tokenize(text)) for text in raw_text]
emb_model = word2vec.Word2Vec(tokenized_texts, sg=1, min_count=1, window=3, vector_size=100)

We can look up the embedding for any given word like this:

In [ ]:
emb_model.wv['happy']

Gensim provides a useful method for finding the most similar words. Internally, this uses cosine similarity to compare embeddings. Let's try it out:

In [ ]:
emb_model.wv.similar_by_word('happy', topn = 5)

Above, we trained our own model using the skipgram method. We can also download a pretrained model that has previously been trained on a large corpus -- this means that we do not need to train our own model, or get hold of a large dataset ourselves. Instead, we can take advantage of a model that was trained on much more data and can provide high-quality embeddings. The idea of sharing pretrained representations of data is really powerful: we don't need to train from scratch for each new task we do, which would be very expensive and require a lot of data; we can just reuse some suitable embeddings to make the task of training much easier and cheaper. 

There is a list of embedding models available [here](https://radimrehurek.com/gensim/models/word2vec.html#pretrained-models). Let's try out GLoVe embeddings. GLoVe is an alternative to the skipgram model. This model was trained on a corpus of tweets, so it may encode different relationships to our previous model. When do you think it would be useful to use embeddings trained on Tweets? Do you think they would be suitable for processing legal documents? 


In [ ]:
import gensim.downloader

glove_wv = gensim.downloader.load('glove-twitter-25')

print(glove_wv['happy'])

**TO-DO 10:** Find the most similar five words to 'happy' according to the GloVe Twitter model.

In [ ]:
# WRITE YOUR OWN CODE HERE


Notice that a different set of words are favoured than with word2vec, and consider how this might result from pretraining the embeddings on Twitter data.

# 6. BERTopic

https://maartengr.github.io/BERTopic/index.html

Rather than relying on sparse vector representations, as in LDA, can we make use of pretrained word embeddings that better capture the meaning of words, to find more meaningful topics? BERTopic is one such approach for this. BERTopic is a pipeline method with several steps:

Embedding documents --> Dimensionality reduction --> Clustering --> \[cluster labelling: Tokenizer --> Token weighting scheme\]

The first step uses a method called SBERT to embed each document to single dense vector. SBERT uses the same principles as the word2vec method above: it uses a large corpus of unlabelled data and trains a model to predict words from their contexts. This results in a model that can produce embeddings of the words in a document. The individual word embeddings are aggregated (usually by taking an average) to form a document embedding. We will learn more about how SBERT works in future weeks. 

Dimensionality reduction (e.g., PCA) is then used to reduce the size of the embeddings further, as clustering works more effectively in lower dimensions. Clustering groups documents with similar embeddings. 

**TO-DO 11**: How does this assignment of documents to clusters differ from the way that LDA assigns topics to documents? 

ANSWER: Write your answer here. 

Each cluster is a collection of documents, and the last two steps attempt to label each cluster. To do this, we tokenize all of the documents in one cluster, then we can count up how many times each term occurs in that cluster. We could take the most frequent words in each cluster as its 'label', but this will include a lot of very common words, so we apply TF-IDF to emphasise the distinctive keywords. 

Let's try out the BERTopic library:

In [ ]:
!pip install bertopic umap-learn  # If you don't have it installed already, install it now

In [ ]:
from bertopic import BERTopic

topic_model = BERTopic()  # run the topic model with default settings. You can also specify the components of the topic model as arguments to the BERTopic constructor, but we will not do that in this lab. See the BERTopic documentation for more details.
topics, probs = topic_model.fit_transform(newsgroups_train.data)

We can display the topics and their labels, ordered by frequency. What do you think topic label '-1' means? Hint: the clustering algorithm is DBScan, which does not place every single document into a cluster.

In [ ]:
topic_model.get_topic_info()

We can also see the top words in each topic:

In [ ]:
topic_model.get_topic(0)

We can also look at information about each document and visualise the relationships between topics:

In [ ]:
topic_model.get_document_info(newsgroups_train.data)

In [ ]:
topic_model.visualize_topics()

**TO-DO 12:** Let's see if BERTopic has produced a strong association between newsgroups and topics. The function below counts up how many documents in each newsgroup are assigned to each topic. Use this to visualise a matrix mapping topics to newsgroups, similar to the one you produced for LDA. Do you see any strong topic associations?

**TO-DO 13:** Referring to the documentation for BERTopic, alter the arguments to the BERTopic constructor to see if you can improve the results. For example, by changing the minimum cluster size. 

In [ ]:
doc_info = topic_model.get_document_info(newsgroups_train.data)
num_topics = len(topic_model.get_topic_info()) - 1  # the number of topics is one less than the number of rows in topic_info, because one row corresponds to outliers which are not assigned to any topic.

def get_bertopic_dists(max_num_docs=10):
    
    count_topics = np.zeros((len(newsgroups_train.target_names), num_topics))  # an array with one entry per topic. 
    
    for t, newsgroup_name in enumerate(newsgroups_train.target_names):
        doc_idxs = get_document_ids_in_newsgroup(newsgroup_name, newsgroups_train)
        
        # only use the first ten documents.
        if len(doc_idxs) > max_num_docs:
            doc_idxs = doc_idxs[:max_num_docs]
        
        for doc_idx in doc_idxs:
            # Get the document from newsgroups_train
            topic_for_doc = doc_info.loc[doc_idx]['Topic']
            if topic_for_doc != -1:  # if the document is not an outlier
                count_topics[t, topic_for_doc] += 1  # increment the count for the topic assigned to this document    

    count_topics /= np.sum(count_topics, axis=1)[:, None]  # normalise the counts to get a distribution over topics for this newsgroup

    return count_topics


count_topics = get_bertopic_dists(max_num_docs=100)

In [ ]:
### WRITE YOUR OWN CODE HERE
def plot_newsgroup_topic_matrix(count_topics):
    
    print(f'count_topics is a matrix of shape {count_topics.shape}')

    # Create a new figure
    plt.figure(figsize=(20,10))
    clear_output()

    # Plot the matrix as a 2-D grid, where colours represent the values.
    plt.imshow(count_topics)

    # Change the labels on the axes
    plt.yticks(range(len(newsgroups_train.target_names)), newsgroups_train.target_names )
    plt.xticks(range(num_topics))

    plt.show()
    
plot_newsgroup_topic_matrix(count_topics)

###

# 7. Optional: Hierarchical Dirichlet Process (HDP)

This section is optional if you want to learn more about HDP as an alternative to LDA. HDP allows the number of topics to grow to fit the dataset, so it is not fixed in advance. 

There is an implementation of the [HDP model provided by gensim](https://radimrehurek.com/gensim/models/hdpmodel.html). Instead of passing in a fixed number of topics, HDP will try to learn a good number of topics to fit the data. It does not always give better results than LDA, but can be effective in some situations.

**OPTIONAL TO-DO 14:** Refer to the documentation for HDP and train an HDP model. Hint: reuse the ```bow_corpus``` and ```dictionary``` as arguments in the same way that you did to construct the ```LdaModel``` object.

Use the trained HDP model to obtain mean topic distributions for each newsgroup in the test set with ```get_newsgroups_mean_topics()```. Plot the mean topic matrix as above and compare it to the results from LDA. Set alpha and gamma to 0.01.

In [ ]:
from gensim.models import HdpModel

### WRITE YOUR OWN CODE HERE



###

# print the word-topic distributions for 
for idx, topic in hdp_model.print_topics(20):
    print("Topic: {} \nWords: {}".format(idx, topic ))
    print("\n")

The previous cell shows the first 20 topics. HDP learns the number of topics that are needed to model the dataset. It produces a global distribution over the topics. Topics with very low probability can be considered inactive:

In [ ]:
def plot_global_topic_weights(hdp_model):
    global_topic_weights = hdp_model.m_varphi_ss / np.sum(hdp_model.m_varphi_ss)

    plt.bar(np.arange(len(global_topic_weights)), global_topic_weights)
    plt.ylabel('Probability')
    plt.xlabel('Topic ID')
    
plot_global_topic_weights(hdp_model)

Run the code below to visualise the topics that HDP finds for each newsgroup:

In [ ]:
plot_newsgroup_topic_matrix(hdp_model, 20)

The alpha and gamma arguments to HdpModel control the 'concentration' of topics. Varying these parameters therefore affects the number of topics that HDP finds -- whether it tends towards many fine-grained topics, or few coarse-grained topics.

**OPTIONAL TO-DO 15:** Change the values of alpha and gamma for the HDP model and see what their effect is. Hint: to see noticable differences, change the values by a factor of 10. 

In [ ]:
### WRITE YOUR OWN CODE HERE

###

# print the word-topic distributions for 
for idx, topic in hdp_model.print_topics(20):
    print("Topic: {} \nWords: {}".format(idx, topic ))
    print("\n")
    
plot_global_topic_weights(hdp_model2)
    

In [ ]:
plot_newsgroup_topic_matrix(hdp_model2, 20)